# Training — experiments as data

**Reads:** `data/<DATASET>/split/`, `configs/augment.json` (optional) · **Writes:** `runs/<run_id>/`.

All training logic lives in `thai_char_cnn.train.fit`; this notebook only says *which* runs to
make and reads their results. Each experiment is one row of overrides on a shared `BASE`.

A run is cached under a hash of its full config, seed, `split_id` and the training code itself
(`code_hash`, over the modules in `src/thai_char_cnn/` that decide what a run produces). Rerunning
the notebook loads finished runs instead of retraining them, so adding a row trains only that row,
and editing the training code invalidates every cached run, so a loaded result always comes from
the code on disk.

Two switches, both off by default:

| env var | effect |
| --- | --- |
| `RETRAIN=1` | ignore the cache and retrain every row (overwrites its run) |
| `VERIFY=1` | §5 retrains the first run into a scratch folder and checks it matches the cached one |
| `EPOCHS=n` | smoke run with `n` epochs (a separate run, since epochs are part of the hash) |

### What the score is, and isn't

- The score is **val macro-F1 over validated classes**: per-class F1 averaged with equal weight, over
  the classes `03_split` could give enough val images. Accuracy is shown too, but with this
  imbalance it mostly reflects the ten biggest classes.
- Val is chosen by the filename-group field named `writer_id`, not a verified person identifier. It
  measures held-out filename groups only, not unseen handwriting, fonts, or a known hidden-test
  distribution. It is also used for early stopping and for choosing between experiments, so it is
  optimistic. The instructor's hidden test set is the real test, and nothing here is called "test".
- Classes flagged *not validated* are trained on, but nothing measures them. They are listed below.

## 1 · Setup and data check

**What:** load the current split and check it passed its own self-checks, then show what the
training pipeline actually feeds the model, augmentation included.

**Why the montage is here:** the augmentation preview (`05_augment_preview.py`) is deferred. Until
it exists, this is the only look at what augmentation does to the glyphs. The defaults are
deliberately mild, and stroke and elastic distortion stay off until someone has looked.

In [1]:
import json
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib import font_manager
from PIL import Image

from thai_char_cnn.augment import build_train_transform, load_augment_config
from thai_char_cnn.data import SplitData
from thai_char_cnn.metrics import top_confused
from thai_char_cnn.paths import CONFIGS, FONT, RAW, SPLIT_DIR
from thai_char_cnn.preprocess import letterbox
from thai_char_cnn.train import DEFAULT_CONFIG, code_hash, fit, load_runs

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)
font_manager.fontManager.addfont(str(FONT))
THAI = font_manager.FontProperties(fname=str(FONT), size=16)

split_run = json.loads((SPLIT_DIR / "run.json").read_text())
SPLIT_ID = split_run["split_id"]
failed = [k for k, v in split_run["checks"].items() if v != "PASS"]
assert not failed, f"03_split self-checks failed: {failed}"

aug_path = CONFIGS / "augment.json"
AUG = load_augment_config(aug_path)
print(f"split_id     : {SPLIT_ID}   ({split_run['generated_at']})")
print(f"augmentation : {'configs/augment.json' if aug_path.exists() else 'defaults (no configs/augment.json yet)'}")
print(f"               {AUG}")
print(f"device       : {'cuda — ' + torch.cuda.get_device_name() if torch.cuda.is_available() else 'cpu'}")

data = SplitData(DEFAULT_CONFIG["img_size"])
print(f"\ntrain images : {len(data.idx['train']):,}")
print(f"val images   : {len(data.idx['val']):,}")
print(f"pixel mean/std (train only): {data.pixel_mean:.4f} / {data.pixel_std:.4f}")
classes = data.classes
print(f"classes      : {classes.status.value_counts().to_dict()}")
print(f"not validated: {', '.join(f'{r.folder} {r.character}' for r in classes[classes.status == 'not_validated'].itertuples())}")
print(f"weak (val<5) : {', '.join(f'{r.folder} {r.character}' for r in classes[classes.status == 'weak'].itertuples())}")

split_id     : 49bb16a21863   (2026-09-23T15:23:59+00:00)
augmentation : defaults (no configs/augment.json yet)
               AugmentConfig(rotation_deg=8.0, shear_deg=5.0, scale_min=0.9, scale_max=1.1, translate_frac=0.04, stroke_p=0.0, elastic_p=0.0, elastic_alpha=8.0, elastic_sigma=3.0)
device       : cuda — NVIDIA GeForce RTX 3060

train images : 48,535
val images   : 12,033
pixel mean/std (train only): 0.6772 / 0.4272
classes      : {'validated': 56, 'not_validated': 11, 'weak': 5}
not validated: 163 ฃ, 177 ฑ, 196 ฤ, 204 ฬ, 206 ฮ, 214 ึ, 244 ๔, 245 ๕, 246 ๖, 247 ๗, 249 ๙
weak (val<5) : 189 ฝ, 207 ฯ, 225 แ, 234 ๊, 248 ๘


In [2]:
# first column: the model input as-is; the rest: independent draws of the train transform
N_ROWS, N_DRAWS = 8, 7
torch.manual_seed(0)
rng = np.random.default_rng(0)
pick = rng.choice(data.idx["train"], N_ROWS, replace=False)
tf = build_train_transform(AUG)
fig, ax = plt.subplots(N_ROWS, N_DRAWS + 1, figsize=(1.2 * (N_DRAWS + 1), 1.25 * N_ROWS))
for r, j in enumerate(pick):
    px = data.pixels[j]
    for c in range(N_DRAWS + 1):
        img = px if c == 0 else tf(px)
        ax[r, c].imshow(img[0], cmap="gray", vmin=0, vmax=255)
        ax[r, c].set_xticks([])
        ax[r, c].set_yticks([])
    ax[r, 0].set_ylabel(classes.character.iloc[int(data.y[j])], fontproperties=THAI, rotation=0, labelpad=14)
ax[0, 0].set_title("input", fontsize=9)
for c in range(1, N_DRAWS + 1):
    ax[0, c].set_title(f"aug {c}", fontsize=9)
plt.tight_layout()
plt.show()

## 2 · Experiment table

`BASE` is shared by every run; `EXPERIMENTS` maps a name to the overrides that make it different.
Anything not listed takes `thai_char_cnn.train.DEFAULT_CONFIG`.

This frozen-split ablation compares aspect-preserving letterbox with square stretch under the
same model and optimization settings. Original geometry is a separate letterbox ablation. Each
configuration uses three seeds because small single-seed gaps are not decision-quality evidence.

`EPOCHS`, `RETRAIN` and `VERIFY` come from the environment (see the top of the notebook), so a
command-line run can set them without editing the notebook.

In [3]:
BASE = dict(epochs=int(os.environ.get("EPOCHS", DEFAULT_CONFIG["epochs"])))
SEEDS = [42, 137, 271]
RETRAIN = os.environ.get("RETRAIN") == "1"
VERIFY = os.environ.get("VERIFY") == "1"

EXPERIMENTS = {
    "letterbox": {},
    "stretch": dict(preprocess="stretch"),
    "letterbox+geometry": dict(use_geometry=True),
}

table = pd.DataFrame({name: DEFAULT_CONFIG | BASE | ov for name, ov in EXPERIMENTS.items()}).T
varying = [c for c in table.columns if table[c].astype(str).nunique() > 1]
print(f"shared      : {({k: v for k, v in (DEFAULT_CONFIG | BASE).items() if k not in varying})}")
print(f"seeds       : {SEEDS}")
print(f"code_hash   : {code_hash()}")
print(f"RETRAIN={RETRAIN}  VERIFY={VERIFY}")
table[varying] if varying else table[[]].assign(overrides="none — baseline only")

shared      : {'model': 'small_cnn', 'width': 32, 'dropout': 0.3, 'img_size': 32, 'augment': False, 'sampler': 'none', 'epochs': 40, 'batch_size': 256, 'lr': 0.003, 'weight_decay': 0.0005, 'warmup_epochs': 1, 'label_smoothing': 0.0, 'patience': 8, 'seed': 42, 'amp': True}
seeds       : [42, 137, 271]
code_hash   : 7d4c909904bf
RETRAIN=False  VERIFY=True


,use_geometry,preprocess
letterbox,False,letterbox
stretch,False,stretch
letterbox+geometry,True,letterbox


## 3 · Train (or load)

One loop. A finished run is loaded from `runs/`; anything new is trained, and with `RETRAIN=1`
everything is. On the RTX 3060 a baseline epoch takes about two seconds.

In [4]:
RUN_DIRS = {}
for name, overrides in EXPERIMENTS.items():
    for seed in SEEDS:
        RUN_DIRS.setdefault(name, []).append(fit(BASE | overrides | dict(name=name, seed=seed),
                                                 force=RETRAIN))

[letterbox] cached -> runs/7908f50ff679
[letterbox] cached -> runs/88dc44f1bf50
[letterbox] cached -> runs/94645ad7d725
[stretch] cached -> runs/43b5e371d294
[stretch] cached -> runs/a94c43441443
[stretch] cached -> runs/6ebfd7931259
[letterbox+geometry] cached -> runs/7c8826fcd7d6
[letterbox+geometry] cached -> runs/6f0381bb835f
[letterbox+geometry] cached -> runs/528a915c7de4


## 4 · Leaderboard

Ranked by val macro-F1 over validated classes, mean ± standard deviation across seeds. Only runs in
the current experiment table are ranked, and those are by construction on the current `split_id`
and `code_hash`. Other runs on disk are listed below as a record, not compared, with the reason: a
different split means a different val set, and different code means a different pipeline.

In [5]:
runs = load_runs()
current_ids = {p.name for dirs in RUN_DIRS.values() for p in dirs}
cur = runs[runs.run_id.isin(current_ids)].copy()
name_of = {p.name: n for n, dirs in RUN_DIRS.items() for p in dirs}
cur["experiment"] = cur.run_id.map(name_of)

board = (cur.groupby("experiment")
         .agg(seeds=("seed", "size"), macro_f1=("val_macro_f1", "mean"), macro_f1_sd=("val_macro_f1", "std"),
              macro_f1_all=("val_macro_f1_all", "mean"), val_acc=("val_acc", "mean"),
              best_epoch=("best_epoch", "mean"), minutes=("train_time_s", lambda s: s.sum() / 60))
         .sort_values("macro_f1", ascending=False).round(4))
print(f"split {SPLIT_ID} · headline = macro-F1 over {int((classes.status == 'validated').sum())} validated classes\n")
display(board)

other = runs[~runs.run_id.isin(current_ids)]
if len(other):
    other = other.assign(reason=np.select(
        [other.split_id != SPLIT_ID, other["cfg.code_hash"] != code_hash()],
        ["older split", "older code"], "not in the experiment table"))
    print(f"\nnot ranked: {len(other)} other run(s) on disk")
    display(other[["run_id", "name", "reason", "split_id", "cfg.code_hash", "cfg.epochs", "seed", "val_macro_f1"]])

split 49bb16a21863 · headline = macro-F1 over 56 validated classes



,seeds,macro_f1,macro_f1_sd,macro_f1_all,val_acc,best_epoch,minutes
experiment,,,,,,,
stretch,3,0.9747,0.0011,0.9727,0.9802,21.3333,2.9883
letterbox+geometry,3,0.9738,0.0030,0.9715,0.9783,18.6667,2.7250
letterbox,3,0.9724,0.0013,0.9681,0.9771,19.3333,2.7533



not ranked: 2 other run(s) on disk


,run_id,name,reason,split_id,cfg.code_hash,cfg.epochs,seed,val_macro_f1
0,05b212628308,baseline,older code,49bb16a21863,NaN,40,42,0.97385
3,6aebfc003412,baseline,older code,49bb16a21863,33800972559e,40,42,0.97385


## 5 · Reproducibility check (`VERIFY=1`)

**What:** retrain the first run of the first experiment from scratch into a scratch folder and
compare it with the cached run: metrics, per-epoch history, every val prediction, and the weights.

**Why:** the cache is only trustworthy if training is deterministic. Seeds, cuDNN's deterministic
mode and seeded data-loader workers are meant to make it so; this measures whether they do.

**How to read it:**

- **exact** — identical weights. The cache is a faithful stand-in for retraining.
- **drift** — macro-F1 within 0.005. Expected after a change of GPU, driver, PyTorch/CUDA version
  or `num_workers`, which reorder floating-point work. Not a bug; worth a note.
- **mismatch** — a larger gap. Something the run hash doesn't capture changed; investigate before
  trusting any cached number.

It costs one extra training run, so it's off unless asked for.

In [6]:
DRIFT_TOL = 0.005
if not VERIFY:
    print("skipped (set VERIFY=1 to retrain and compare)")
else:
    ref_name = next(iter(EXPERIMENTS))
    ref = RUN_DIRS[ref_name][0]
    ref_cfg = BASE | EXPERIMENTS[ref_name] | dict(name=ref_name, seed=SEEDS[0])
    with tempfile.TemporaryDirectory() as tmp:
        new = fit(ref_cfg, data=data, runs_dir=Path(tmp), verbose=False)
        ma, mb = (json.loads((d / "metrics.json").read_text()) for d in (ref, new))
        ha, hb = (pd.read_csv(d / "history.csv") for d in (ref, new))
        pa, pb = (pd.read_csv(d / "val_predictions.csv") for d in (ref, new))
        sa, sb = (torch.load(d / "model.pt", weights_only=False)["state_dict"] for d in (ref, new))
    same_id = new.name == ref.name
    dw = max(float((sa[k].float() - sb[k].float()).abs().max()) for k in sa)
    df1 = abs(ma["val_macro_f1"] - mb["val_macro_f1"])
    hcols = ["train_loss", "val_loss", "val_macro_f1"]
    verdict = ("exact" if same_id and dw == 0 else "drift" if same_id and df1 <= DRIFT_TOL else "mismatch")

    print(f"retrained {ref_name!r} (run {ref.name}) from scratch in {mb['train_time_s']:.0f}s")
    print(f"  same run id          : {same_id}")
    print(f"  val macro-F1         : cached {ma['val_macro_f1']:.6f}  retrained {mb['val_macro_f1']:.6f}")
    print(f"  best epoch           : cached {ma['best_epoch']}  retrained {mb['best_epoch']}")
    print(f"  history max |diff|   : {float((ha[hcols] - hb[hcols]).abs().max().max()) if len(ha) == len(hb) else 'length differs'}")
    print(f"  val predictions diff : {int((pa.pred_idx != pb.pred_idx).sum())} of {len(pa):,}")
    print(f"  weights max |diff|   : {dw}")
    print(f"\nverdict: {verdict.upper()}")
    assert verdict != "mismatch", "retraining does not reproduce the cached run"

retrained 'letterbox' (run 7908f50ff679) from scratch in 81s
  same run id          : True
  val macro-F1         : cached 0.973850  retrained 0.973850
  best epoch           : cached 33  retrained 33
  history max |diff|   : 0.0
  val predictions diff : 0 of 12,033
  weights max |diff|   : 0.0

verdict: EXACT


## 6 · Deep dive

Everything below reads the run named by `FOCUS` (its first seed). Change `FOCUS` to look at another
experiment; nothing else needs editing.

- **Per-class F1**, weakest first. Grey bars are *weak* classes (fewer than 5 val images); their F1 is
  close to noise. Not-validated classes have no val images and are absent.
- **Top confused pairs.** These feed the next ruling pass: a pair confused in both directions is
  either genuinely similar glyphs or a labelling problem, and only looking at the images says which.
- **Error gallery**: the most confident mistakes, which are where label errors tend to hide.

In [7]:
FOCUS = "letterbox"
run_dir = RUN_DIRS[FOCUS][0]
m = json.loads((run_dir / "metrics.json").read_text())
hist = pd.read_csv(run_dir / "history.csv")
pc = pd.read_csv(run_dir / "per_class.csv", keep_default_na=False, na_values=[""])
cm = np.load(run_dir / "confusion.npy")
preds = pd.read_csv(run_dir / "val_predictions.csv")

print(f"{FOCUS}  run {m['run_id']}  split {m['split_id']}")
print(f"  val macro-F1 (validated classes) : {m['val_macro_f1']:.4f}")
print(f"  val macro-F1 (all with val)      : {m['val_macro_f1_all']:.4f}")
print(f"  val accuracy                     : {m['val_acc']:.4f}")
print(f"  best epoch {m['best_epoch']} of {m['epochs_run']} run, {m['train_time_s']:.0f}s, {m['n_params']:,} params")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))
ax[0].plot(hist.epoch, hist.train_loss, label="train")
ax[0].plot(hist.epoch, hist.val_loss, label="val")
ax[0].set(title="Loss", xlabel="epoch")
ax[0].legend()
ax[1].plot(hist.epoch, hist.val_macro_f1, color="#C44E52", label="val macro-F1")
ax[1].plot(hist.epoch, hist.val_acc, color="#8C8C8C", label="val accuracy")
ax[1].axvline(m["best_epoch"], ls="--", c="gray", lw=1)
ax[1].set(title="Validation", xlabel="epoch")
ax[1].legend()
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

letterbox  run 7908f50ff679  split 49bb16a21863
  val macro-F1 (validated classes) : 0.9739
  val macro-F1 (all with val)      : 0.9727
  val accuracy                     : 0.9811
  best epoch 33 of 40 run, 82s, 296,168 params


In [8]:
shown = pc[pc.support > 0].sort_values("f1")
fig, ax = plt.subplots(figsize=(15, 3.8))
ax.bar(range(len(shown)), shown.f1, color=np.where(shown.status == "weak", "#BBBBBB", "#4C72B0"))
ax.set_xticks(range(len(shown)))
ax.set_xticklabels(shown.character, fontproperties=THAI)
ax.set(title=f"{FOCUS}: val F1 per class, weakest first (grey = weak)", ylabel="F1", ylim=(0, 1.02))
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("weakest 12 classes:")
shown.head(12)[["class_idx", "folder", "character", "status", "train_n", "support", "precision", "recall", "f1"]].round(3)

weakest 12 classes:


,class_idx,folder,character,status,train_n,support,precision,recall,f1
70,70,248,๘,weak,24,3,1.000,0.667,0.800
63,63,241,๑,validated,39,7,1.000,0.714,0.833
48,48,216,ุ,validated,136,18,0.833,0.833,0.833
44,44,212,ิ,validated,36,11,1.000,0.818,0.900
65,65,243,๓,validated,17,6,1.000,0.833,0.909
47,47,215,ื,validated,53,7,1.000,0.857,0.923
53,53,227,ใ,validated,877,205,0.935,0.912,0.923
43,43,210,า,validated,3673,943,0.934,0.924,0.929
33,33,199,ว,validated,1343,357,0.914,0.952,0.933
55,55,229,ๅ,validated,1036,379,0.956,0.918,0.937


In [9]:
ch = classes.set_index("class_idx")
tc = top_confused(cm, 15)
tc["true"] = tc.true_idx.map(lambda i: f"{ch.folder[i]} {ch.character[i]}")
tc["predicted"] = tc.pred_idx.map(lambda i: f"{ch.folder[i]} {ch.character[i]}")
tc["reverse_count"] = [int(cm[p, t]) for t, p in zip(tc.true_idx, tc.pred_idx, strict=True)]
tc[["true", "predicted", "count", "share_of_true", "reverse_count"]].round(3)

,true,predicted,count,share_of_true,reverse_count
0,210 า,199 ว,32,0.034,14
1,229 ๅ,210 า,31,0.082,16
2,227 ใ,210 า,16,0.078,9
3,210 า,229 ๅ,16,0.017,31
4,199 ว,210 า,14,0.039,32
5,233 ้,209 ั,9,0.047,1
6,180 ด,181 ต,9,0.020,5
7,210 า,227 ใ,9,0.010,16
8,183 ท,190 พ,5,0.015,1
9,181 ต,180 ด,5,0.016,9


In [10]:
N_ERR = 32
err = preds[preds.true_idx != preds.pred_idx].sort_values("p_pred", ascending=False).head(N_ERR)
print(f"val errors: {int((preds.true_idx != preds.pred_idx).sum()):,} of {len(preds):,} — the {len(err)} most confident:")
cols = 8
fig, ax = plt.subplots(int(np.ceil(len(err) / cols)), cols, figsize=(1.6 * cols, 1.9 * np.ceil(len(err) / cols)),
                       squeeze=False)
for a in ax.ravel():
    a.axis("off")
for a, r in zip(ax.ravel(), err.itertuples(), strict=False):
    with Image.open(RAW / r.path) as im:
        a.imshow(letterbox(im.convert("L"), 48), cmap="gray", vmin=0, vmax=255)
    a.set_title(f"{ch.character[r.true_idx]} as {ch.character[r.pred_idx]}  {r.p_pred:.2f}", fontproperties=THAI)
plt.tight_layout()
plt.show()
err.assign(true=err.true_idx.map(ch.folder), pred=err.pred_idx.map(ch.folder))[
    ["path", "writer_id", "true", "pred", "p_pred", "p_true"]].head(12)

val errors: 228 of 12,033 — the 32 most confident:


,path,writer_id,true,pred,p_pred,p_true
101,161/bc_020sg_3_334.jpg,bc_020,161,205,1.0,0.0
1585,170/be_016sg_11_83.jpg,be_016,170,162,1.0,0.0
499,162/be_002sg_10_162.jpg,be_002,162,209,1.0,0.0
348,161/be_006sg_8_257.jpg,be_006,161,182,1.0,0.0
2085,180/be_016sg_11_199.jpg,be_016,180,181,1.0,0.0
1540,170/be_006sg_4_84.jpg,be_006,170,162,1.0,0.0
9829,210/bc_020tg_3_447.jpg,bc_020,210,199,1.0,0.0
6941,197/be_002tg_7_119.jpg,be_002,197,182,1.0,0.0
5814,194/be_018tg_11_26.jpg,be_018,194,201,1.0,0.0
5839,194/be_018tg_7_188.jpg,be_018,194,201,1.0,0.0
